# RelCheck v2 — Viability Test (30 images)
## GroundingDINO (Spatial) + Llama-4-Maverick VLM (Action/Attribute)

**Goal:** Test the new type-aware verification pipeline on 30 images.
- Spatial triples → GroundingDINO bbox geometry (deterministic)
- Action/attribute triples → Llama-4-Maverick VLM + multi-question voting
- Uses images already cached on Google Drive (no re-downloading)

**Decision gate (Cell 9):** Are corrections sensible on ≥60% of corrected images?
- YES → Build full pipeline for 600-image evaluation
- NO → Evaluate which component failed and pivot

In [ ]:
# ============================================================
# Cell 1 — Install Dependencies + Setup
# ============================================================
# Runtime: ~2 min on Colab T4

!pip install -q together transformers pillow torch torchvision
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

import os, json, base64, re, time, textwrap
from io import BytesIO
from collections import defaultdict
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from together import Together
from transformers import (
    Blip2Processor, Blip2ForConditionalGeneration,
    AutoProcessor, AutoModelForZeroShotObjectDetection,
)

# ── API Key ──
TOGETHER_API_KEY = ""  # <-- paste your key here
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
client = Together(api_key=TOGETHER_API_KEY)

# ── Model IDs ──
VLM_MODEL   = "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"   # vision
LLM_MODEL   = "meta-llama/Llama-3.3-70B-Instruct-Turbo"             # text-only
GDINO_MODEL = "IDEA-Research/grounding-dino-tiny"                    # detection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"VLM:    {VLM_MODEL}")
print(f"LLM:    {LLM_MODEL}")
print(f"GDino:  {GDINO_MODEL}")
print("Setup complete.")

In [ ]:
# ============================================================
# Cell 2 — Load Models (BLIP-2 + GroundingDINO on GPU)
# ============================================================
# Runtime: ~3 min (downloading weights first time)

print("Loading BLIP-2 (captioner)...")
blip2_processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-flan-t5-xl", torch_dtype=torch.float16
).to(DEVICE)
print("  BLIP-2 loaded.")

print("Loading GroundingDINO (detector)...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_MODEL)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    GDINO_MODEL
).to(DEVICE)
print("  GroundingDINO loaded.")

print("All local models loaded.")

In [ ]:
# ============================================================
# Cell 3 — Load 30 Test Images (from Google Drive cache)
# ============================================================
# Uses images already downloaded by the Master notebook.
# NO re-downloading — only uses what's already on Drive.

from google.colab import drive
from pathlib import Path
import random

# Mount Google Drive (will prompt if not already mounted)
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"

# ── Check what's available ──
if not os.path.isdir(DRIVE_IMAGES_DIR):
    print(f"ERROR: Image cache not found at {DRIVE_IMAGES_DIR}")
    print("Run the Master notebook (Cells 0-22) first to download images.")
    raise FileNotFoundError(DRIVE_IMAGES_DIR)

cached_images = list(Path(DRIVE_IMAGES_DIR).glob("*.jpg")) + list(Path(DRIVE_IMAGES_DIR).glob("*.jpeg"))
print(f"Found {len(cached_images)} cached images on Drive.")

if len(cached_images) < 20:
    print(f"WARNING: Only {len(cached_images)} images cached. Need at least 20.")
    print("Run the Master notebook image download cell first.")

# ── Load R-Bench annotations if available (for diverse selection) ──
rbench_loaded = False
if os.path.exists(RBENCH_PATH):
    with open(RBENCH_PATH) as f:
        rbench_data = json.load(f)
    # Get image IDs that have R-Bench questions (so we can evaluate later)
    rbench_image_ids = set()
    for entry in rbench_data:
        img_field = entry.get("image", entry.get("image_id", ""))
        if img_field:
            img_id = os.path.splitext(os.path.basename(str(img_field)))[0]
            rbench_image_ids.add(img_id)
    print(f"R-Bench covers {len(rbench_image_ids)} unique images.")
    rbench_loaded = True
else:
    print("R-Bench annotations not found — will select random cached images.")
    rbench_image_ids = set()

# ── Select 30 images: prefer those with R-Bench annotations ──
NUM_IMAGES = 30
random.seed(42)  # reproducible

cached_id_to_path = {}
for p in cached_images:
    img_id = p.stem  # filename without extension
    cached_id_to_path[img_id] = str(p)

if rbench_loaded:
    # Prefer images that have R-Bench questions (so we can evaluate)
    annotated = [iid for iid in cached_id_to_path if iid in rbench_image_ids]
    unannotated = [iid for iid in cached_id_to_path if iid not in rbench_image_ids]
    random.shuffle(annotated)
    random.shuffle(unannotated)
    selected_ids = annotated[:NUM_IMAGES]
    if len(selected_ids) < NUM_IMAGES:
        selected_ids += unannotated[:NUM_IMAGES - len(selected_ids)]
else:
    all_ids = list(cached_id_to_path.keys())
    random.shuffle(all_ids)
    selected_ids = all_ids[:NUM_IMAGES]

print(f"\nSelected {len(selected_ids)} images for viability test.")

# ── Load images ──
images = {}
for img_id in selected_ids:
    path = cached_id_to_path[img_id]
    try:
        images[img_id] = Image.open(path).convert("RGB")
    except Exception as e:
        print(f"  Failed to open {img_id}: {e}")

print(f"Loaded {len(images)} images successfully.")

# Show first 8 as preview
preview_ids = list(images.keys())[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_id in zip(axes.flat, preview_ids):
    ax.imshow(images[img_id])
    ax.set_title(img_id[:12], fontsize=9)
    ax.axis("off")
for ax in axes.flat[len(preview_ids):]:
    ax.axis("off")
plt.suptitle(f"Preview: 8 of {len(images)} test images", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Cell 4 — Stage 1: BLIP-2 Caption Generation
# ============================================================

captions = {}
for img_id, img in images.items():
    inputs = blip2_processor(images=img, return_tensors="pt").to(DEVICE, torch.float16)
    with torch.no_grad():
        out = blip2_model.generate(**inputs, max_new_tokens=80)
    caption = blip2_processor.decode(out[0], skip_special_tokens=True).strip()
    captions[img_id] = caption
    print(f"[{img_id}] {caption}")

print(f"\nGenerated {len(captions)} captions.")

In [ ]:
# ============================================================
# Cell 5 — Stage 2: Triple Extraction + Type Classification
# ============================================================

TRIPLE_EXTRACTION_PROMPT = """Extract ALL relational triples from this caption as JSON.

Each triple: {{"subject": "...", "relation": "...", "object": "...", "type": "SPATIAL|ACTION|ATTRIBUTE"}}

Type rules:
- SPATIAL: physical position (on, in, above, below, next to, behind, near, under, between, etc.)
- ACTION: interaction/activity (holding, riding, eating, wearing, sitting on, walking, playing with, using, with, etc.)
- ATTRIBUTE: property/quality (is red, is large, is wooden, looks old, etc.)

Caption: "{caption}"

Return ONLY a JSON array of triples. No explanation."""

all_triples = {}
for img_id, caption in captions.items():
    prompt = TRIPLE_EXTRACTION_PROMPT.format(caption=caption)
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=500,
    )
    raw = resp.choices[0].message.content.strip()
    # Parse JSON (handle markdown code blocks)
    json_str = raw
    if "```" in json_str:
        json_str = json_str.split("```")[1]
        if json_str.startswith("json"):
            json_str = json_str[4:]
    try:
        triples = json.loads(json_str.strip())
    except json.JSONDecodeError:
        print(f"  [{img_id}] JSON parse failed, raw: {raw[:200]}")
        triples = []

    all_triples[img_id] = triples
    print(f"\n[{img_id}] Caption: {caption}")
    for t in triples:
        print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> {t['type']}")

total = sum(len(v) for v in all_triples.values())
print(f"\nExtracted triples for {len(all_triples)} images. Total: {total}")


In [ ]:
# ============================================================
# Cell 6 — Stage 3a: GroundingDINO Detection + Spatial Geometry
# ============================================================

DETECTION_THRESHOLD = 0.25

def detect_objects(image, text_queries, threshold=DETECTION_THRESHOLD):
    """Detect objects using GroundingDINO. Returns list of (label, score, bbox_norm)."""
    text = ". ".join(text_queries) + "."
    inputs = gdino_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)

    results = gdino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=threshold,
        text_threshold=threshold,
        target_sizes=[image.size[::-1]],  # (H, W)
    )[0]

    detections = []
    W, H = image.size
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        x1, y1, x2, y2 = box.tolist()
        bbox_norm = [x1/W, y1/H, x2/W, y2/H]
        detections.append((label, score.item(), bbox_norm))
    return detections


def check_spatial_relation(subj_box, obj_box, relation):
    """
    Check if a spatial relation holds between two bounding boxes.
    Boxes are [x1, y1, x2, y2] normalized to [0, 1].
    y increases downward (image coordinates).
    Returns: True (supported), False (contradicted), None (can't determine)
    """
    sx1, sy1, sx2, sy2 = subj_box
    ox1, oy1, ox2, oy2 = obj_box

    s_cx, s_cy = (sx1+sx2)/2, (sy1+sy2)/2
    o_cx, o_cy = (ox1+ox2)/2, (oy1+oy2)/2

    # Intersection
    ix1, iy1 = max(sx1, ox1), max(sy1, oy1)
    ix2, iy2 = min(sx2, ox2), min(sy2, oy2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    area_s = max((sx2-sx1) * (sy2-sy1), 1e-8)
    area_o = max((ox2-ox1) * (oy2-oy1), 1e-8)
    union = area_s + area_o - inter
    iou = inter / union if union > 0 else 0

    containment = inter / area_s  # fraction of subject inside object

    rel = relation.lower().strip()

    if rel in ("on", "on top of", "atop"):
        # "on" = subject resting on object surface.
        # Accept if: (a) subject centroid is above object centroid, OR
        # (b) significant overlap (subject bbox overlaps object's top region).
        # Must have horizontal overlap.
        h_overlap = max(0, min(sx2, ox2) - max(sx1, ox1))
        h_denom = min(sx2-sx1, ox2-ox1)
        h_ratio = h_overlap / h_denom if h_denom > 1e-8 else 0
        if h_ratio < 0.15:
            return False  # no horizontal overlap → can't be "on"
        # Subject bottom should be near or above object top, OR strong overlap
        subj_bottom_near_obj_top = abs(sy2 - oy1) < 0.15  # within 15% of image height
        centroid_above = s_cy < o_cy
        strong_overlap = iou > 0.1
        return (centroid_above and h_ratio > 0.2) or subj_bottom_near_obj_top or strong_overlap

    elif rel in ("above", "over"):
        return s_cy < o_cy

    elif rel in ("below", "under", "beneath", "underneath"):
        return s_cy > o_cy

    elif rel in ("next to", "beside", "near"):
        distance = ((s_cx - o_cx)**2 + (s_cy - o_cy)**2) ** 0.5
        return distance < 0.5 and iou < 0.3

    elif rel in ("in", "inside"):
        return containment > 0.5

    elif rel in ("left of", "to the left of", "to the left"):
        return s_cx < o_cx

    elif rel in ("right of", "to the right of", "to the right"):
        return s_cx > o_cx

    elif rel in ("behind", "in front of", "between"):
        return None  # can't determine from 2D bboxes

    else:
        return None  # unknown → fall through to VLM


def find_best_detection(detections, query):
    """
    Find the highest-confidence detection matching query.
    Uses word-boundary matching to avoid 'man' matching 'woman'.
    """
    query_lower = query.lower().strip()
    query_words = set(query_lower.split())
    best = None
    best_score = -1

    for label, score, bbox in detections:
        label_lower = label.lower().strip()
        label_words = set(label_lower.split())

        # Exact match (best)
        if query_lower == label_lower:
            if score > best_score:
                best = (label, score, bbox)
                best_score = score
            continue

        # Word overlap match (e.g., "red car" matches "car")
        # But "man" should NOT match "woman"
        if query_words & label_words:  # at least one shared word
            if score > best_score:
                best = (label, score, bbox)
                best_score = score
            continue

        # Substring match only if query is multi-word or label is single-word
        # This avoids "man" in "woman" but allows "dining table" to match "table"
        if len(query_lower) > 4 and (query_lower in label_lower or label_lower in query_lower):
            if score > best_score:
                best = (label, score, bbox)
                best_score = score

    return best


# ── Run spatial verification ──
spatial_results = {}

for img_id, triples in all_triples.items():
    img = images[img_id]
    results = []

    all_queries = set()
    for t in triples:
        all_queries.add(t["subject"].lower())
        all_queries.add(t["object"].lower())

    detections = detect_objects(img, list(all_queries))
    print(f"\n[{img_id}] Detected {len(detections)} objects: {[(d[0], round(d[1],2)) for d in detections]}")

    for t in triples:
        if t["type"] != "SPATIAL":
            results.append((t, "SKIP", "Not spatial — will use VLM"))
            continue

        subj_det = find_best_detection(detections, t["subject"])
        obj_det = find_best_detection(detections, t["object"])

        if subj_det is None or obj_det is None:
            missing = []
            if subj_det is None: missing.append(t["subject"])
            if obj_det is None: missing.append(t["object"])
            results.append((t, "FALLBACK_VLM", f"Detection failed for: {missing}"))
            print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> FALLBACK (missing: {missing})")
            continue

        geo_result = check_spatial_relation(subj_det[2], obj_det[2], t["relation"])
        if geo_result is None:
            results.append((t, "FALLBACK_VLM", f"No geometry rule for '{t['relation']}'"))
            print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> FALLBACK (no geo rule)")
        elif geo_result:
            evidence = f"Subj bbox={[round(x,3) for x in subj_det[2]]}, Obj bbox={[round(x,3) for x in obj_det[2]]}. Geometry confirms '{t['relation']}'"
            results.append((t, "SUPPORTED", evidence))
            print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> SUPPORTED (geometry)")
        else:
            evidence = f"Subj bbox={[round(x,3) for x in subj_det[2]]}, Obj bbox={[round(x,3) for x in obj_det[2]]}. Geometry CONTRADICTS '{t['relation']}'"
            results.append((t, "HALLUCINATED", evidence))
            print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> HALLUCINATED (geometry)")

    spatial_results[img_id] = results

# Visualize detections for first image
if images:
    first_id = list(images.keys())[0]
    img = images[first_id]
    all_q = set()
    for t in all_triples[first_id]:
        all_q.add(t["subject"].lower())
        all_q.add(t["object"].lower())
    dets = detect_objects(img, list(all_q))

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)
    W, H = img.size
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(dets), 1)))
    for i, (label, score, bbox) in enumerate(dets):
        x1, y1, x2, y2 = bbox[0]*W, bbox[1]*H, bbox[2]*W, bbox[3]*H
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                  edgecolor=colors[i], facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1-5, f"{label} ({score:.2f})", color=colors[i],
                fontsize=10, fontweight="bold", backgroundcolor="black")
    ax.set_title(f"[{first_id}] GroundingDINO Detections")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# Cell 7 — Stage 3b: VLM Verification (Action/Attribute)
#          Llama-4-Maverick + Multi-Question Voting
# ============================================================

def encode_image_base64(image):
    """Convert PIL Image to base64 string."""
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def generate_paraphrases(subject, relation, obj, triple_type):
    """Generate 3 paraphrased yes/no questions for a triple."""
    s, r, o = subject, relation, obj

    if triple_type == "ACTION":
        return [
            f"Is the {s} {r} the {o}?",
            f"Does the {s} appear to be {r} the {o}?",
            f"Can you see the {s} {r} the {o} in this image?",
        ]
    elif triple_type == "ATTRIBUTE":
        # Handle "is/are" relations: avoid "Is the dress is red?"
        # Strip the copula from the relation for natural questions
        r_stripped = re.sub(r'^(is|are|was|were|be)\s*', '', r.lower()).strip()
        if r_stripped:
            # e.g., (dress, "is red", _) or (dress, "is", "red")
            attr = r_stripped if not o else f"{r_stripped} {o}".strip()
        else:
            attr = o
        return [
            f"Is the {s} {attr}?",
            f"Does the {s} look {attr}?",
            f"Would you describe the {s} as {attr}?",
        ]
    else:  # spatial fallback or OTHER
        return [
            f"Is the {s} {r} the {o}?",
            f"In this image, is the {s} {r} the {o}?",
            f"Does the image show the {s} {r} the {o}?",
        ]


def vlm_verify_triple(image, questions, max_retries=2):
    """
    Send image + questions to Llama-4-Maverick.
    Returns (yes_ratio, raw_answers).
    """
    b64 = encode_image_base64(image)
    answers = []

    for q in questions:
        prompt = f"{q} Answer with ONLY 'yes' or 'no'."
        for attempt in range(max_retries + 1):
            try:
                resp = client.chat.completions.create(
                    model=VLM_MODEL,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {
                                "url": f"data:image/jpeg;base64,{b64}"
                            }},
                        ],
                    }],
                    temperature=0.0,
                    max_tokens=10,
                )
                answer = resp.choices[0].message.content.strip().lower()
                if "yes" in answer:
                    answers.append(1.0)
                elif "no" in answer:
                    answers.append(0.0)
                else:
                    answers.append(0.5)  # ambiguous
                break
            except Exception as e:
                if attempt < max_retries:
                    time.sleep(2)
                else:
                    print(f"    VLM error after {max_retries+1} attempts: {e}")
                    answers.append(0.5)

        time.sleep(0.3)  # rate limit courtesy

    yes_ratio = np.mean(answers) if answers else 0.5
    return yes_ratio, answers


# ── Thresholds ──
VLM_SUPPORTED_THRESHOLD = 0.65
VLM_HALLUCINATED_THRESHOLD = 0.40

# ── Run VLM verification ──
vlm_results = {}

for img_id in all_triples:
    img = images[img_id]
    results = []
    spatial_res = spatial_results.get(img_id, [])

    # Collect triples needing VLM
    needs_vlm = []
    for t, verdict, evidence in spatial_res:
        if verdict in ("SKIP", "FALLBACK_VLM"):
            needs_vlm.append((t, evidence))

    print(f"\n[{img_id}] VLM verification for {len(needs_vlm)} triples")

    for t, fallback_reason in needs_vlm:
        questions = generate_paraphrases(t["subject"], t["relation"], t["object"], t["type"])
        print(f"    Questions: {questions}")  # debug: verify grammar
        yes_ratio, raw = vlm_verify_triple(img, questions)

        if yes_ratio >= VLM_SUPPORTED_THRESHOLD:
            verdict = "SUPPORTED"
            evidence = f"VLM yes_ratio={yes_ratio:.2f} ({raw}). Relation '{t['relation']}' confirmed."
        elif yes_ratio < VLM_HALLUCINATED_THRESHOLD:
            verdict = "HALLUCINATED"
            evidence = f"VLM yes_ratio={yes_ratio:.2f} ({raw}). Relation '{t['relation']}' NOT confirmed."
        else:
            verdict = "UNCERTAIN"
            evidence = f"VLM yes_ratio={yes_ratio:.2f} ({raw}). Inconclusive."

        results.append((t, verdict, evidence))
        print(f"  ({t['subject']}, {t['relation']}, {t['object']}) -> {verdict} (yes_ratio={yes_ratio:.2f})")

    vlm_results[img_id] = results

print("\n=== VLM Verification Complete ===")


In [ ]:
# ============================================================
# Cell 8 — Stage 4: Merge All Verdicts + Correction
# ============================================================

CORRECTION_PROMPT = """You are a precise caption editor. Fix ONLY the specific hallucinated relations listed below.

Original caption: "{caption}"

Hallucinated triples (with evidence):
{hallucination_list}

Rules:
1. Change ONLY the words related to the hallucinated relations
2. Keep everything else EXACTLY the same
3. If you cannot determine what the correct relation should be, REMOVE the hallucinated claim rather than guessing
4. Output ONLY the corrected caption — no explanation

Corrected caption:"""


def merge_verdicts(img_id):
    """Combine spatial geometry + VLM results into final verdicts."""
    merged = []
    spatial = spatial_results.get(img_id, [])
    vlm = vlm_results.get(img_id, [])

    vlm_lookup = {}
    for t, v, e in vlm:
        key = (t["subject"], t["relation"], t["object"])
        vlm_lookup[key] = (v, e)

    for t, verdict, evidence in spatial:
        key = (t["subject"], t["relation"], t["object"])
        if verdict in ("SUPPORTED", "HALLUCINATED"):
            merged.append((t, verdict, evidence))
        elif verdict in ("SKIP", "FALLBACK_VLM"):
            if key in vlm_lookup:
                v, e = vlm_lookup[key]
                merged.append((t, v, e))
            else:
                merged.append((t, "UNCERTAIN", "No verification available"))
    return merged


def correct_caption(caption, hallucinated_triples):
    """Use Llama-3.3-70B to correct hallucinated spans."""
    if not hallucinated_triples:
        return caption, "No hallucinations found"

    hall_list = ""
    for i, (t, evidence) in enumerate(hallucinated_triples, 1):
        hall_list += f"{i}. ({t['subject']}, {t['relation']}, {t['object']}) — {evidence}\n"

    prompt = CORRECTION_PROMPT.format(caption=caption, hallucination_list=hall_list)
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=200,
    )
    corrected = resp.choices[0].message.content.strip()
    # Strip wrapping quotes
    if corrected.startswith('"') and corrected.endswith('"'):
        corrected = corrected[1:-1]
    if corrected.startswith("'") and corrected.endswith("'"):
        corrected = corrected[1:-1]
    return corrected, hall_list


def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(curr[-1]+1, prev[j+1]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]


# ── Run merge + correction ──
final_results = {}

print("=" * 70)
print("RELCHECK v2 — VIABILITY TEST RESULTS")
print("=" * 70)

for img_id in images:
    caption = captions[img_id]
    merged = merge_verdicts(img_id)

    hallucinated = [(t, e) for t, v, e in merged if v == "HALLUCINATED"]
    supported = [(t, e) for t, v, e in merged if v == "SUPPORTED"]
    uncertain = [(t, e) for t, v, e in merged if v == "UNCERTAIN"]

    if hallucinated:
        corrected, hall_details = correct_caption(caption, hallucinated)
    else:
        corrected = caption
        hall_details = "None"

    edit_rate = levenshtein(caption, corrected) / max(len(caption), 1)

    final_results[img_id] = {
        "caption": caption,
        "corrected": corrected,
        "triples": len(merged),
        "supported": len(supported),
        "hallucinated": len(hallucinated),
        "uncertain": len(uncertain),
        "edit_rate": edit_rate,
        "details": merged,
    }

    print(f"\n{'─'*60}")
    print(f"[{img_id}]")
    print(f"  Original:     {caption}")
    print(f"  Corrected:    {corrected}")
    print(f"  Triples: {len(merged)} | Supported: {len(supported)} | "
          f"Hallucinated: {len(hallucinated)} | Uncertain: {len(uncertain)}")
    print(f"  Edit rate: {edit_rate:.1%}")
    if hallucinated:
        for t, e in hallucinated:
            print(f"    X ({t['subject']}, {t['relation']}, {t['object']})")


In [ ]:
# ============================================================
# Cell 9 — DECISION GATE: Viability Verdict
# ============================================================

print("\n" + "=" * 70)
print("VIABILITY ASSESSMENT — 30 IMAGE TEST")
print("=" * 70)

total_images = len(final_results)
images_with_corrections = sum(1 for r in final_results.values() if r["hallucinated"] > 0)
images_no_change = sum(1 for r in final_results.values() if r["hallucinated"] == 0)
total_triples = sum(r["triples"] for r in final_results.values())
total_supported = sum(r["supported"] for r in final_results.values())
total_hallucinated = sum(r["hallucinated"] for r in final_results.values())
total_uncertain = sum(r["uncertain"] for r in final_results.values())

print(f"\n  Images tested:           {total_images}")
print(f"  Images with corrections: {images_with_corrections}/{total_images} ({images_with_corrections/max(total_images,1):.0%})")
print(f"  Images unchanged:        {images_no_change}/{total_images}")
print(f"  Total triples:           {total_triples}")
print(f"  Supported:               {total_supported} ({total_supported/max(total_triples,1):.0%})")
print(f"  Hallucinated:            {total_hallucinated} ({total_hallucinated/max(total_triples,1):.0%})")
print(f"  Uncertain:               {total_uncertain} ({total_uncertain/max(total_triples,1):.0%})")
print(f"  Avg triples/image:       {total_triples/max(total_images,1):.1f}")

# ── Quality checks per image ──
print(f"\n--- Per-Image Quality Checks ---")
bad_images = []
for img_id, r in final_results.items():
    issues = []
    if r["edit_rate"] > 0.5:
        issues.append(f"edit_rate={r['edit_rate']:.0%}")
    if r["triples"] > 0 and r["hallucinated"] == r["triples"]:
        issues.append("ALL triples hallucinated")
    if r["hallucinated"] > 0 and r["corrected"] == r["caption"]:
        issues.append("correction identical to original")
    if issues:
        bad_images.append(img_id)
        print(f"  WARNING [{img_id}]: {', '.join(issues)}")

good_corrections = images_with_corrections - len([
    img_id for img_id in bad_images
    if final_results[img_id]["hallucinated"] > 0
])

print(f"\n--- Verification Strategy Breakdown ---")
geo_supported = sum(1 for img_id, rl in spatial_results.items()
                    for t, v, e in rl if v == "SUPPORTED")
geo_hallucinated = sum(1 for img_id, rl in spatial_results.items()
                       for t, v, e in rl if v == "HALLUCINATED")
geo_fallback = sum(1 for img_id, rl in spatial_results.items()
                   for t, v, e in rl if v == "FALLBACK_VLM")
vlm_total = sum(len(rl) for rl in vlm_results.values())
print(f"  Geometry SUPPORTED:    {geo_supported}")
print(f"  Geometry HALLUCINATED: {geo_hallucinated}")
print(f"  Geometry → VLM fallback: {geo_fallback}")
print(f"  VLM verified triples:  {vlm_total}")

print(f"\n--- Correction Quality ---")
edit_rates = [r["edit_rate"] for r in final_results.values() if r["hallucinated"] > 0]
if edit_rates:
    print(f"  Avg edit rate (corrected only): {np.mean(edit_rates):.1%}")
    print(f"  Max edit rate: {max(edit_rates):.1%}")
    print(f"  Min edit rate: {min(edit_rates):.1%}")
else:
    print("  No corrections made.")

# ── Show 5 example corrections ──
print(f"\n--- Example Corrections (up to 5) ---")
corrected_items = [(img_id, r) for img_id, r in final_results.items() if r["hallucinated"] > 0]
for img_id, r in corrected_items[:5]:
    print(f"\n  [{img_id}]")
    print(f"    Before: {r['caption']}")
    print(f"    After:  {r['corrected']}")
    print(f"    Edit rate: {r['edit_rate']:.1%} | Hallucinated: {r['hallucinated']}/{r['triples']}")

# ── Final verdict ──
viable_threshold = 0.6  # 60% of corrected images should be sensible
if images_with_corrections > 0:
    quality_ratio = good_corrections / images_with_corrections
else:
    quality_ratio = 1.0  # no corrections needed = pipeline is conservative (fine)

print(f"\n{'='*70}")
print(f"  Good corrections: {good_corrections}/{images_with_corrections} ({quality_ratio:.0%})")
print(f"  Bad corrections:  {len([x for x in bad_images if final_results[x]['hallucinated'] > 0])}")

if quality_ratio >= viable_threshold and total_triples > 0:
    print(f"\n  VERDICT: VIABLE — proceed to full pipeline build")
    print(f"  Next: Build relcheck v2 modules, run 600-image evaluation")
elif total_hallucinated == 0:
    print(f"\n  VERDICT: INCONCLUSIVE — no hallucinations detected")
    print(f"  The pipeline may be too conservative. Consider lowering thresholds:")
    print(f"    VLM_SUPPORTED_THRESHOLD: {VLM_SUPPORTED_THRESHOLD} -> 0.55")
    print(f"    VLM_HALLUCINATED_THRESHOLD: {VLM_HALLUCINATED_THRESHOLD} -> 0.35")
else:
    print(f"\n  VERDICT: NOT VIABLE — too many bad corrections")
    print(f"  Investigate: Are GroundingDINO detections correct? Is Maverick answering sensibly?")
print("=" * 70)

# Save results
with open("viability_v2_results.json", "w") as f:
    save = {}
    for img_id, r in final_results.items():
        save[img_id] = {
            "caption": r["caption"],
            "corrected": r["corrected"],
            "triples": r["triples"],
            "supported": r["supported"],
            "hallucinated": r["hallucinated"],
            "uncertain": r["uncertain"],
            "edit_rate": r["edit_rate"],
        }
    json.dump(save, f, indent=2)
print("\nResults saved to viability_v2_results.json")
